In [ ]:
from pathlib import Path

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image as PILImage
import matplotlib.pyplot as plt


# ============================================================
# PATH HELPERS
# ============================================================

def find_existing_path(possible_paths):
    """
    Return the first path that exists from a list of possible paths.
    """
    for path in possible_paths:
        path = Path(path)
        if path.exists():
            return path
    return None


# -----------------------------
# Data paths
# -----------------------------
table_path = find_existing_path([
    "/storage1/fs1/jmillman/Active/DigitalTwin/Notebooks/7_StreamLit/TableS14_PS_Exo_Endo.xlsx",
    "7_StreamLit/TableS14_PS_Exo_Endo.xlsx",
    "TableS14_PS_Exo_Endo.xlsx"
])

ko_fig_dir = find_existing_path([
    "/storage1/fs1/jmillman/Active/DigitalTwin/Notebooks/7_StreamLit/KO_fig",
    "7_StreamLit/KO_fig",
    "KO_fig"
])

cell_state_fig_path = find_existing_path([
    "/storage1/fs1/jmillman/Active/DigitalTwin/Notebooks/7_StreamLit/Cell_State_fig/umap_DT_PP_Exo_EP_yeslegend_noaxes.png",
    "7_StreamLit/Cell_State_fig/umap_DT_PP_Exo_EP_yeslegend_noaxes.png",
    "Cell_State_fig/umap_DT_PP_Exo_EP_yeslegend_noaxes.png"
])

expression_fig_dir = find_existing_path([
    "/storage1/fs1/jmillman/Active/DigitalTwin/Notebooks/7_StreamLit/Expression_fig",
    "7_StreamLit/Expression_fig",
    "Expression_fig"
])

if table_path is None:
    raise FileNotFoundError("Could not find TableS14_PS_Exo_Endo.xlsx")

if ko_fig_dir is None:
    raise FileNotFoundError("Could not find KO_fig folder")

if cell_state_fig_path is None:
    raise FileNotFoundError("Could not find cell-state UMAP figure")

if expression_fig_dir is None:
    raise FileNotFoundError("Could not find Expression_fig folder")


# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_excel(table_path, sheet_name="Table S14", header=1)
df = df.loc[:, ~df.columns.astype(str).str.contains("^Unnamed")]
df.columns = df.columns.astype(str).str.strip()
df["TF_KO"] = df["TF_KO"].astype(str)

# Rename columns to match preferred cell-state labels
rename_dict = {
    "PPExo": "PP-Exo",
    "PPeEP": "PP-eEP"
}
df = df.rename(columns=rename_dict)

# Desired cell-state order for this subset
cell_state_order = ["Exo", "PP-Exo", "PP", "PP-eEP", "eEP"]

# Keep only states present in the table, in the desired order
cell_state_cols = [state for state in cell_state_order if state in df.columns]

if len(cell_state_cols) == 0:
    raise ValueError("No expected cell-state columns found in the table.")


# ============================================================
# STYLE AND CONSTANTS
# ============================================================

pal = {
    "Mesench.": "#ffbd00",
    "UE": "#EE3377",
    "DE": "#a7defa",
    "GT": "#DDCC77",
    "PP": "#402bad",
    "PP-Exo": "#07C0F0",
    "PP-eEP": "#BD27A3",
    "Exo": "#117733",
    "eEP": "#fcb686",
    "lEP": "#cd853f",
    "Alpha": "#f55e07",
    "Delta": "#7d9e31",
    "EC": "#015f94",
    "Beta": "#ba0c2f"
}

dropdown_layout = widgets.Layout(width="420px")
dropdown_style = {"description_width": "120px"}

# Global Y-axis limits for all TF KO plots
global_y_min = df[cell_state_cols].min().min()
global_y_max = df[cell_state_cols].max().max()
global_y_range = global_y_max - global_y_min
global_y_air = global_y_range * 0.05 if global_y_range != 0 else 0.001

global_y_min_plot = global_y_min - global_y_air
global_y_max_plot = global_y_max + global_y_air


# ============================================================
# DISPLAY HELPERS
# ============================================================

def find_ko_figure(tf_name):
    """
    Finds a KO figure using expected file name:
    Plot {TF} KO.jpg
    """
    possible_extensions = [".jpg", ".jpeg", ".png", ".tif", ".tiff"]

    for ext in possible_extensions:
        candidate = ko_fig_dir / f"Plot {tf_name} KO{ext}"
        if candidate.exists():
            return candidate

    return None


def find_expression_figure(tf_name):
    """
    Finds an expression figure using expected file name:
    umap_DT_PP_Exo_EP_{TF}_nolegend_noaxis.png
    """
    possible_extensions = [".png", ".jpg", ".jpeg", ".tif", ".tiff"]

    for ext in possible_extensions:
        candidate = expression_fig_dir / f"umap_DT_PP_Exo_EP_{tf_name}_nolegend_noaxis{ext}"
        if candidate.exists():
            return candidate

    return None


def display_resized_image(path, width=300, rotate_left=False):
    """
    Opens an image, optionally rotates it 90 degrees left,
    converts RGBA/transparency safely to RGB, and displays it at a fixed width.
    """
    img = PILImage.open(path)

    if rotate_left:
        img = img.rotate(90, expand=True)

    # Convert transparent images safely for notebook display
    if img.mode in ("RGBA", "LA"):
        background = PILImage.new("RGB", img.size, "white")
        background.paste(img, mask=img.split()[-1])
        img = background
    else:
        img = img.convert("RGB")

    # Resize while preserving aspect ratio
    w, h = img.size
    new_height = int(h * (width / w))
    img = img.resize((width, new_height))

    display(img)


# ============================================================
# HEADER
# ============================================================

header = widgets.HTML("""
<h1>Digital Twin Perturbation Score Explorer</h1>
<h2>Endocrine-exocrine branching subset</h2>
<p>
Explore precomputed transcription factor knockout perturbation scores 
across pancreatic progenitor, exocrine, and endocrine progenitor cell states:
<b>Exo</b>, Exocrine; 
<b>PP-Exo</b>, Intermediate Exocrine; 
<b>PP</b>, Pancreatic Progenitor; 
<b>PP-eEP</b>, Intermediate Early Endocrine Progenitor; and 
<b>eEP</b>, Early Endocrine Progenitor.
</p>
""")


# ============================================================
# WIDGET 1: Top TF KOs effects per Cell State
# ============================================================

cell_state_dropdown = widgets.Dropdown(
    options=cell_state_cols,
    description="Cell state:",
    layout=dropdown_layout,
    style=dropdown_style
)

top_n_slider = widgets.IntSlider(
    value=10,
    min=5,
    max=50,
    step=1,
    description="Top N:",
    layout=dropdown_layout,
    style=dropdown_style
)

out_rank_tables = widgets.Output()

def update_rank_tables(change=None):
    with out_rank_tables:
        clear_output(wait=True)

        selected_cell_state = cell_state_dropdown.value
        top_n = top_n_slider.value

        state_data = df[["TF_KO", selected_cell_state]].copy()
        state_data = state_data.rename(columns={selected_cell_state: "Perturbation score"})

        top_positive = (
            state_data
            .sort_values("Perturbation score", ascending=False)
            .head(top_n)
            .reset_index(drop=True)
        )

        top_negative = (
            state_data
            .sort_values("Perturbation score", ascending=True)
            .head(top_n)
            .reset_index(drop=True)
        )

        positive_out = widgets.Output()
        negative_out = widgets.Output()

        with positive_out:
            display(widgets.HTML(f"<h4>Top positive TF KO effects for {selected_cell_state}</h4>"))
            display(top_positive)

        with negative_out:
            display(widgets.HTML(f"<h4>Top negative TF KO effects for {selected_cell_state}</h4>"))
            display(top_negative)

        display(widgets.HBox(
            [positive_out, negative_out],
            layout=widgets.Layout(gap="40px", align_items="flex-start")
        ))

section_1 = widgets.VBox([
    widgets.HTML("<h3>1. Top TF KOs effects per Cell State</h3>"),
    cell_state_dropdown,
    top_n_slider,
    out_rank_tables
])

cell_state_dropdown.observe(update_rank_tables, names="value")
top_n_slider.observe(update_rank_tables, names="value")


# ============================================================
# WIDGET 2: Perturbation Scores per Gene KO in each Cell State
# ============================================================

tf_dropdown = widgets.Dropdown(
    options=sorted(df["TF_KO"].unique()),
    description="TF KO:",
    layout=dropdown_layout,
    style=dropdown_style
)

out_tf_plot = widgets.Output()

def update_tf_plot(change=None):
    with out_tf_plot:
        clear_output(wait=True)

        selected_tf = tf_dropdown.value

        tf_row = df[df["TF_KO"] == selected_tf][cell_state_cols].iloc[0]
        tf_row = tf_row.reindex(cell_state_cols)

        bar_colors = [pal.get(state, "#999999") for state in tf_row.index]

        plt.figure(figsize=(7, 4.5))
        plt.bar(tf_row.index, tf_row.values, color=bar_colors)
        plt.axhline(0, color="black", linewidth=0.8)
        plt.ylim(global_y_min_plot, global_y_max_plot)

        plt.ylabel("Perturbation score")
        plt.title(f"{selected_tf} KO perturbation profile across cell states")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()

section_2 = widgets.VBox([
    widgets.HTML("<h3>2. Perturbation Scores per Gene KO in each Cell State</h3>"),
    tf_dropdown,
    out_tf_plot
])

tf_dropdown.observe(update_tf_plot, names="value")


# ============================================================
# WIDGET 3: TF KO simulation figure
# ============================================================

ko_tf_dropdown = widgets.Dropdown(
    options=sorted(df["TF_KO"].unique()),
    description="TF KO:",
    layout=dropdown_layout,
    style=dropdown_style
)

out_ko_fig = widgets.Output()

def update_ko_figure(change=None):
    with out_ko_fig:
        clear_output(wait=True)

        selected_tf = ko_tf_dropdown.value
        ko_fig_path = find_ko_figure(selected_tf)
        expression_fig_path = find_expression_figure(selected_tf)

        display(widgets.HTML(
            f"<h4>{selected_tf} KO simulation compared with cell-state map and RNA expression</h4>"
        ))

        missing_messages = []

        if ko_fig_path is None:
            missing_messages.append(
                f"No KO figure found for {selected_tf}. Expected: Plot {selected_tf} KO.[jpg/png/tif]"
            )

        if expression_fig_path is None:
            missing_messages.append(
                f"No expression figure found for {selected_tf}. Expected: umap_DT_PP_Exo_EP_{selected_tf}_nolegend_noaxis.png"
            )

        if missing_messages:
            for message in missing_messages:
                print(message)
            return

        static_out = widgets.Output()
        ko_out = widgets.Output()
        expr_out = widgets.Output()

        with static_out:
            display(widgets.HTML("<b>Cell-state map</b>"))
            display_resized_image(cell_state_fig_path, width=300, rotate_left=False)

        with ko_out:
            display(widgets.HTML(f"<b>{selected_tf} KO simulation</b>"))
            display_resized_image(ko_fig_path, width=300, rotate_left=True)

        with expr_out:
            display(widgets.HTML(f"<b>{selected_tf} RNA expression</b>"))
            display_resized_image(expression_fig_path, width=300, rotate_left=True)

        display(widgets.HBox(
            [static_out, ko_out, expr_out],
            layout=widgets.Layout(gap="20px", align_items="flex-start")
        ))

section_3 = widgets.VBox([
    widgets.HTML("<h3>3. TF KO simulation figure</h3>"),
    ko_tf_dropdown,
    out_ko_fig
])

ko_tf_dropdown.observe(update_ko_figure, names="value")


# ============================================================
# DISPLAY DASHBOARD
# ============================================================

dashboard = widgets.VBox([
    header,
    section_1,
    widgets.HTML("<hr>"),
    section_2,
    widgets.HTML("<hr>"),
    section_3
])

display(dashboard)

# Initialize widgets
update_rank_tables()
update_tf_plot()
update_ko_figure()